# RetainAI — Block 4: Tool-Using LLM Agent on Amazon Bedrock

The agent ties every earlier block together. For a customer (and optionally their message) it:

1. Pulls the profile and a **live churn score from the production SageMaker endpoint** (Block 2), with SHAP reasons (Block 1)
2. Classifies the customer's message with the **intent model** (Block 3)
3. Calls the **plan engine**: company policy plus the Block 5 bandit decide the resolution step, the offer, and
   whether to escalate, based on the customer's risk *and* their message. It can also search the **policy and FAQ (RAG)**
4. Writes the reply: resolve the issue first, then present the offer

Then we **evaluate** it like a production system: programmatic guardrail checks, LLM-as-judge quality scores,
latency, and token usage, comparing two models.

**Design principle:** code and learned models decide *what to do*; the LLM decides *how to say it*.
An earlier version let the LLM choose the offer, and testing showed it made tone-deaf choices (for example,
pushing a contract on a customer reporting a billing error). Moving decisions into the plan engine fixed that.

In [ ]:
%pip install -q shap xgboost scikit-learn

## 1. Setup

In [ ]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
sys.path.insert(0, "src")
from agent import Toolbox, Agent, validate, judge, AGENT_MODEL, JUDGE_MODEL, CHALLENGER_MODEL, STRONG_MODEL

tools = Toolbox()
active = tools.customers[tools.customers["Churn"] == 0]
print(f"Loaded {len(tools.customers):,} customers | endpoint: {tools.endpoint} | policy sections: {len(tools.policy_chunks)}")
print("Agent:", AGENT_MODEL, "| Strong:", STRONG_MODEL, "| Challenger:", CHALLENGER_MODEL, "| Judge:", JUDGE_MODEL)

## 2. Test each tool directly (before any LLM is involved)

In [ ]:
cid = active.sort_values("revenue_at_risk_12m", ascending=False).index[0]
show = lambda name, out: print(f"--- {name}\n{json.dumps(out, indent=2, default=str)}\n")

show("get_customer_profile", tools.get_customer_profile(cid))
show("score_churn_risk (live endpoint)", tools.score_churn_risk(cid))
show("classify_message", tools.classify_message("I want to cancel, the internet keeps dropping"))
show("recommend_offer (billing message)", tools.recommend_offer(cid, "My bill looks wrong this month."))
show("search_policy", tools.search_policy("where can I find my invoice"))

## 3. Run the agent once and inspect its reasoning trace

In [ ]:
agent = Agent(tools, model_id=AGENT_MODEL)
result = agent.run(cid, "My bill looks wrong this month.")

for step in result["trace"]:
    print(f"🔧 {step['tool']}({json.dumps(step['input'])})")
print(f"\nTurns: {result['turns']} | Latency: {result['latency_s']}s | Tokens: {result['usage']}\n")
print(json.dumps(result["output"], indent=2))
print("\nGuardrail checks:", validate(result, tools))

## 4. Evaluation scenarios

A mix of risk levels and situations, including cases where the right answer is *not* to offer a discount.

In [ ]:
by_risk = active.sort_values("churn_prob", ascending=False)
high = by_risk[by_risk["churn_prob"] >= 0.6].index
mid = by_risk[(by_risk["churn_prob"] >= 0.3) & (by_risk["churn_prob"] < 0.6)].index
low = by_risk[by_risk["churn_prob"] < 0.15].index
anomaly = active[(active["billing_anomaly"] == 1) & (active["churn_prob"] >= 0.3)].index
no_internet = active[(active["InternetService"] == "No") & (active["churn_prob"] >= 0.3)].index

def first(idx, n=0):
    return idx[n] if len(idx) > n else by_risk.index[n]

scenarios = [
    {"name": "High risk, proactive outreach", "customer_id": first(high, 1), "message": None},
    {"name": "High risk, price complaint", "customer_id": first(high, 2),
     "message": "My bill keeps going up and the service hasn't changed. Why am I paying this much?"},
    {"name": "High risk, wants to close account", "customer_id": first(high, 3),
     "message": "Third outage this month. I want to close my account."},
    {"name": "High risk, double charge", "customer_id": first(high, 4),
     "message": "You charged me twice this month and nobody has helped me."},
    {"name": "Medium risk, proactive", "customer_id": first(mid, 0), "message": None},
    {"name": "Medium risk, simple question", "customer_id": first(mid, 1),
     "message": "Where can I find my latest invoice?"},
    {"name": "Low risk, proactive (should be NO_OFFER)", "customer_id": first(low, 0), "message": None},
    {"name": "Low risk, simple question", "customer_id": first(low, 1),
     "message": "How do I update my payment method?"},
    {"name": "Billing anomaly flagged", "customer_id": first(anomaly, 0),
     "message": "My bill looks wrong this month."},
    {"name": "Low risk, says they're leaving (override)", "customer_id": first(low, 2),
     "message": "I'm switching providers next month, please cancel my service."},
    {"name": "Phone-only customer at risk", "customer_id": first(no_internet, 0), "message": None},
]
pd.DataFrame(scenarios)

## 5. Evaluate the agent model

In [ ]:
def evaluate(model_id, scenarios):
    runner, rows, runs = Agent(tools, model_id=model_id), [], []
    for s in scenarios:
        res = runner.run(s["customer_id"], s["message"])
        checks = validate(res, tools)
        scores = judge(res, tools) if res["output"] else {}
        out = res["output"] or {}
        rows.append({"scenario": s["name"], "resolution": out.get("resolution"), "offer": out.get("offer_code"),
                     "escalate": out.get("escalate_to_human"),
                     **{k: v for k, v in checks.items()}, "judge_avg": scores.get("judge_avg"),
                     "judge_comment": scores.get("comment"), "latency_s": res["latency_s"],
                     "input_tokens": res["usage"]["inputTokens"], "output_tokens": res["usage"]["outputTokens"],
                     "tool_calls": len(res["trace"])})
        runs.append(res)
        print(f"{'✅' if checks['compliant'] else '❌'} {s['name']:<42} {out.get('resolution')} + {out.get('offer_code')}  judge={scores.get('judge_avg')}")
    return pd.DataFrame(rows), runs

agent_df, agent_runs = evaluate(AGENT_MODEL, scenarios)
agent_df.drop(columns=["judge_comment"])

In [ ]:
# Read a few of the actual messages the agent wrote
for res in agent_runs[:4]:
    out = res["output"] or {}
    print(f"[{res['customer_id']}] {res['message'] or '(proactive)'}")
    print(f"  resolution: {out.get('resolution')} | offer: {out.get('offer_code')} | escalate: {out.get('escalate_to_human')}")
    print(f"  -> {out.get('customer_message')}\n")

## 6. Model comparison: does a bigger or smaller model change the outcome?

In [ ]:
strong_df, strong_runs = evaluate(STRONG_MODEL, scenarios)
challenger_df, challenger_runs = evaluate(CHALLENGER_MODEL, scenarios)

def summarize(df, model):
    return {"model": model.split(".")[-1], "compliance_rate": df["compliant"].mean(),
            "avg_judge_score": df["judge_avg"].mean(), "avg_latency_s": df["latency_s"].mean(),
            "avg_input_tokens": df["input_tokens"].mean(), "avg_output_tokens": df["output_tokens"].mean(),
            "avg_tool_calls": df["tool_calls"].mean()}

comparison = pd.DataFrame([summarize(agent_df, AGENT_MODEL), summarize(strong_df, STRONG_MODEL),
                           summarize(challenger_df, CHALLENGER_MODEL)])
comparison.round(3)

In [ ]:
# Which guardrails failed, per model?
check_cols = ["valid_json", "called_required_tools", "offer_is_eligible", "follows_plan", "resolution_mentioned",
              "discount_within_policy", "no_internal_terms", "no_invented_perks", "escalation_correct", "under_120_words"]
pd.DataFrame({AGENT_MODEL.split(".")[-1]: agent_df[check_cols].mean(),
              STRONG_MODEL.split(".")[-1]: strong_df[check_cols].mean(),
              CHALLENGER_MODEL.split(".")[-1]: challenger_df[check_cols].mean()}).round(2)

## 7. Save results and example transcripts

In [ ]:
os.makedirs("artifacts", exist_ok=True); os.makedirs("docs", exist_ok=True)
agent_df.assign(model=AGENT_MODEL).to_csv("artifacts/agent_eval.csv", index=False)
challenger_df.assign(model=CHALLENGER_MODEL).to_csv("artifacts/agent_eval_challenger.csv", index=False)
strong_df.assign(model=STRONG_MODEL).to_csv("artifacts/agent_eval_strong.csv", index=False)
json.dump(comparison.round(4).to_dict(orient="records"), open("artifacts/agent_comparison.json", "w"), indent=2)

lines = ["# RetainAI Agent — Example Transcripts\n", f"Model: `{AGENT_MODEL}`\n"]
for s, res in zip(scenarios, agent_runs):
    out = res["output"] or {}
    lines += [f"## {s['name']}\n", f"**Customer message:** {s['message'] or '_(proactive outreach)_'}\n",
              "**Tool calls:** " + " → ".join(t["tool"] for t in res["trace"]) + "\n",
              f"**Resolution:** `{out.get('resolution')}` | **Offer:** `{out.get('offer_code')}` | **Escalate:** {out.get('escalate_to_human')}\n",
              f"**Internal reasons:** {', '.join(out.get('key_reasons', []))}\n",
              f"> {out.get('customer_message')}\n"]
open("docs/agent_examples.md", "w").write("\n".join(lines))
print(comparison.round(3).to_string(index=False))